# JAX-ALFA Simulation Run Log

Scans every `run.log` file under `examples*/` and reports when each simulation
was run, how long it took, and on which platform. Sort by **Completed** ascending
to identify the oldest (potentially stale) runs.

In [1]:
from IPython.display import display, Markdown
from datetime import datetime, timezone
display(Markdown(f"*Last run: {datetime.now(timezone.utc).strftime('%B %d, %Y at %H:%M UTC')}*"))

*Last run: June 24, 2026 at 09:42 UTC*

In [2]:
import os, re
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone, timedelta
from IPython.display import display, HTML

def find_repo_root():
    path = Path.cwd().resolve()
    for candidate in (path, *path.parents):
        if (candidate / 'examples').is_dir() and (candidate / 'docs').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate jaxalfa repository root')

def read_log_info(log_path, tail_bytes=16384):
    """Read elapsed time and CFL values from the tail of a run.log."""
    info = {'elapsed': None, 'cfl_last': None, 'cfl_max': None}
    try:
        size = os.path.getsize(log_path)
        with open(log_path, 'rb') as f:
            f.seek(max(0, size - tail_bytes))
            tail = f.read().decode('utf-8', errors='ignore')
        m = re.search(r'Total Elapsed Time:\s+([\d.]+)\s+seconds', tail)
        if m:
            info['elapsed'] = float(m.group(1))
        for m in re.finditer(r'Current CFL:\s+([\d.]+)', tail):
            info['cfl_last'] = float(m.group(1))
        for m in re.finditer(r'CFLmax:\s+([\d.]+)', tail):
            info['cfl_max'] = float(m.group(1))
    except Exception:
        pass
    return info

repo = find_repo_root()
print(f'Repository root: {repo}')

Repository root: /Users/sukantabasu/Dropbox/Codes/LES/JAX-ALFA/jaxalfa


In [3]:
rows = []

for log_path in sorted(repo.glob('examples*/*/runs/*/run.log')):
    rel      = log_path.relative_to(repo).parts
    platform = rel[0]
    case     = rel[1]
    run      = rel[3]

    mtime     = os.path.getmtime(log_path)
    completed = datetime.fromtimestamp(mtime, tz=timezone.utc)
    info      = read_log_info(log_path)
    elapsed   = info['elapsed']

    if elapsed is not None:
        started   = completed - timedelta(seconds=elapsed)
        wall_time = str(timedelta(seconds=int(elapsed)))
    else:
        started   = None
        wall_time = '—'

    rows.append({
        'Platform'       : platform,
        'Case'           : case,
        'Run'            : run,
        'Started (UTC)'  : started.strftime('%Y-%m-%d %H:%M') if started else '—',
        'Completed (UTC)': completed.strftime('%Y-%m-%d %H:%M'),
        'Wall Time'      : wall_time,
        'CFL (last)'     : f"{info['cfl_last']:.4f}" if info['cfl_last'] is not None else '—',
        'CFL (max)'      : f"{info['cfl_max']:.4f}"  if info['cfl_max']  is not None else '—',
        '_completed_dt'  : completed,
    })

df_all = pd.DataFrame(rows)
print(f'{len(df_all)} runs found across {df_all["Platform"].nunique()} platform(s)')

98 runs found across 4 platform(s)


## By Completion Date (oldest first)

Oldest runs appear at the top — these are candidates for re-running with the latest model.

In [4]:
display_cols = ['Platform', 'Case', 'Run', 'Started (UTC)', 'Completed (UTC)', 'Wall Time', 'CFL (last)', 'CFL (max)']

df_by_date = (
    df_all.sort_values('_completed_dt')
          .reset_index(drop=True)
          [display_cols]
)
df_by_date.index += 1
display(df_by_date.style.set_properties(**{'text-align': 'left'}))

,Platform,Case,Run,Started (UTC),Completed (UTC),Wall Time
1,examples,SBL_GABLS1,64x64x64_LAD_SM_SP,2026-05-22 13:28,2026-05-22 13:51,0:23:01
2,examples,SBL_GABLS1,64x64x64_LASDD_WL_SP,2026-05-22 13:27,2026-05-22 13:52,0:24:23
3,examples,SBL_GABLS1,64x64x64_LASDD_SM_SP,2026-05-22 13:27,2026-05-22 13:54,0:26:47
4,examples,SBL_GABLS1,64x64x64_LAD_SM_DP,2026-05-22 13:28,2026-05-22 14:09,0:40:29
5,examples,SBL_GABLS1,64x64x64_LASDD_WL_DP,2026-05-22 13:27,2026-05-22 14:10,0:42:10
6,examples,SBL_GABLS1,64x64x64_LASDD_SM_DP,2026-05-22 13:27,2026-05-22 14:12,0:45:04
7,examples,SBL_GABLS1,64x64x64_LAD_WL_SP,2026-05-22 16:31,2026-05-22 16:53,0:22:31
8,examples,SBL_GABLS1,64x64x64_LAD_WL_DP,2026-05-22 16:31,2026-05-22 17:10,0:39:24
9,examples,SBL_GABLS1,128x128x128_LASDD_WL_SP,2026-05-22 16:36,2026-05-22 19:44,3:08:26
10,examples,SBL_GABLS1,128x128x128_LASDD_SM_SP,2026-05-22 16:35,2026-05-22 19:52,3:16:59


## By Platform → Case → Run

Same data grouped alphabetically by platform, case, and run name.

In [5]:
df_by_name = (
    df_all.sort_values(['Platform', 'Case', 'Run'])
          .reset_index(drop=True)
          [display_cols]
)
df_by_name.index += 1
display(df_by_name.style.set_properties(**{'text-align': 'left'}))

,Platform,Case,Run,Started (UTC),Completed (UTC),Wall Time
1,examples,CBL_N91,128x128x128_LAD_SM_DP,2026-06-09 08:35,2026-06-09 08:52,0:17:55
2,examples,CBL_N91,128x128x128_LAD_SM_SP,2026-06-09 08:34,2026-06-09 08:46,0:11:48
3,examples,CBL_N91,128x128x128_LAD_WL_DP,2026-06-09 08:35,2026-06-09 08:51,0:16:29
4,examples,CBL_N91,128x128x128_LAD_WL_SP,2026-06-09 08:35,2026-06-09 08:46,0:11:24
5,examples,CBL_N91,128x128x128_LASDD_SM_DP,2026-06-09 08:33,2026-06-09 08:55,0:21:28
6,examples,CBL_N91,128x128x128_LASDD_SM_SP,2026-06-09 08:33,2026-06-09 08:48,0:14:52
7,examples,CBL_N91,128x128x128_LASDD_WL_DP,2026-06-09 08:34,2026-06-09 08:53,0:18:43
8,examples,CBL_N91,128x128x128_LASDD_WL_SP,2026-06-09 08:34,2026-06-09 08:47,0:13:31
9,examples,CBL_N91,256x256x256_LAD_SM_DP,2026-06-09 11:05,2026-06-09 13:10,2:05:03
10,examples,CBL_N91,256x256x256_LAD_SM_SP,2026-06-09 11:04,2026-06-09 12:14,1:09:43


## Summary by Platform

In [6]:
summary = (
    df_all.groupby('Platform')
          .agg(
              Runs        = ('Run', 'count'),
              Oldest_Run  = ('_completed_dt', lambda x: x.min().strftime('%Y-%m-%d')),
              Newest_Run  = ('_completed_dt', lambda x: x.max().strftime('%Y-%m-%d')),
          )
          .rename(columns={'Oldest_Run': 'Oldest Run', 'Newest_Run': 'Newest Run'})
)
display(summary)

,Runs,Oldest Run,Newest Run
Platform,,,
examples,90,2026-05-22,2026-06-18
examples_A6000,1,2026-05-22,2026-05-22
examples_A6000ada,5,2026-06-14,2026-06-17
examples_sandbox_A6000ada,2,2026-05-31,2026-06-01
